# Fine-tuning de Laya multilingüe con CodiEsp, segunda vuelta (Kaggle GPU T4 ×2)

Adaptación del notebook oficial `laya_finetune_typed_decisions_2xT4_kaggle.ipynb`
(Convai Innovations, Apache 2.0) para entrenar **laya-multilingual** con casos clínicos
en español del corpus **CodiEsp**, exportados desde Delphi con `TLayaTrainingExporter`.

Preguntas: `diabetes`, `hipertension`, `cancer`, `insuf_renal` (noul) y `tabaco` (choice).

**Datos:** CodiEsp, Barcelona Supercomputing Center, licencia CC BY 4.0.
Miranda-Escalada A, Gonzalez-Agirre A, Armengol-Estapé J, Krallinger M. *Overview of automatic
clinical coding: annotations, guidelines, and solutions for non-English clinical cases at CodiEsp
track of CLEF eHealth 2020.* CLEF (Working Notes), 2020. https://doi.org/10.5281/zenodo.3837305

---

### Antes de ejecutar
1. **Sube `codiesp.jsonl` como dataset de Kaggle**: panel derecho → *Input* → *Upload* → *New Dataset*,
   con el nombre que quieras (por ejemplo `codiesp-laya`). El notebook lo busca solo dentro de `/kaggle/input`.
2. **Notebook options** (panel derecho): *Accelerator* = **GPU T4 x2**, *Internet* = **On**.
3. Mejor que *Run All*: **Save Version → Save & Run All (Commit)**. Se ejecuta en segundo plano aunque cierres
   el navegador, y los archivos quedan guardados en la pestaña *Output* de la versión.
   Al final tendrás `laya_codiesp_v2_model.zip` (unos 600 MB) para descargar.

### Cambios respecto al original
* Modelo base: `convaiinnovations/laya-multilingual` (el inglés no lee español).
* `max_len` = 2048 tokens (el original usa 1024): algunos casos clínicos son largos.
* Micro-batch 4 × acumulación 8 (mismo batch efectivo de 64), para que 2048 tokens quepan en una T4.
* **Sobremuestreo de la clase minoritaria** en entrenamiento: hay pocos positivos
  (p. ej. 61 diabéticos de 750) y sin esto el modelo aprendería a contestar siempre "no".
* Evaluación propia sobre la partición `test` de CodiEsp, con precisión y exhaustividad por pregunta.

### Segunda vuelta
* Datos exportados con `LabelSmoothing = 0.05` (el preprocesado lo comprueba y lo muestra).
* Modelo `laya-codiesp-v2` en `/kaggle/working/laya_codiesp_v2`.
* La evaluación muestra también la **calibración**: confianza media de aciertos y de errores.
* El zip ya no incluye el punto de control intermedio.


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 3. Cargar `codiesp.jsonl` y preprocesar
Cada fila del archivo es un caso con sus 5 preguntas. Se convierten en secuencias tokenizadas
(una por pregunta) con las mismas funciones de `laya` que usa el notebook oficial.
Solo se usan las filas `train`; las `test` se guardan para la evaluación final.

In [ ]:
import os, json, glob, collections, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya-multilingual"
MAX_LEN = 2048        # tokens de estado + pregunta
HEAD_MAX_LEN = 256    # tokens de pregunta y opciones (como el original)

# --- localizar el archivo subido como dataset ---
found = glob.glob("/kaggle/input/**/codiesp.jsonl", recursive=True) or \
        glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
assert found, "No encuentro codiesp.jsonl en /kaggle/input: súbelo como dataset (ver instrucciones arriba)."
DATA_FILE = found[0]
print("Datos:", DATA_FILE)

rows = [json.loads(l) for l in open(DATA_FILE, encoding="utf-8") if l.strip()]

# comprobar el suavizado de etiquetas del archivo (0.05 esperado en la segunda vuelta)
_g = next(iter(json.loads(rows[0]["gold"]).values()))
_smooth = 1 - max(_g["probabilities"].values())
print(f"Suavizado de etiquetas detectado: ~{_smooth * 2 if _g['type'] == 'noul' else _smooth:.3f}"
      "  (0 = etiquetas absolutas)")
train_rows = [r for r in rows if r["split"] == "train"]
test_rows = [r for r in rows if r["split"] == "test"]
print(f"Casos: {len(rows)}  (train {len(train_rows)}, test {len(test_rows)})")

print(f"Descargando {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)
print("max_len del modelo base:", cfg.get("max_len"), "-> usamos", MAX_LEN)
cfg["max_len"] = MAX_LEN
cfg["head_max_len"] = HEAD_MAX_LEN

def build_training_item(state, qid, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit},
                                  cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t],
            "target": target, "label": label, "qid": qid}

items, dropped = [], 0
for row in train_rows:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, qid, q, gold[qid])
            if it:
                items.append(it)
            else:
                dropped += 1

lens = sorted(len(it["ids"]) for it in items)
at_max = sum(1 for n in lens if n >= MAX_LEN)
print(f"Secuencias de entrenamiento: {len(items)} (descartadas: {dropped})")
print(f"Longitud en tokens: mediana {lens[len(lens)//2]}, máxima {lens[-1]}, en el límite {at_max}")

counts = collections.Counter((it["qid"], it["label"]) for it in items)
print("\nEtiquetas en train (índice de la opción: 0/1 = false/true en noul):")
for key in sorted(counts):
    print(f"  {key[0]:<14} {key[1]}  {counts[key]}")

torch.save(items, "/kaggle/working/train_items.pt")
with open("/kaggle/working/test_rows.json", "w", encoding="utf-8") as f:
    json.dump(test_rows, f, ensure_ascii=False)
print("\nGuardado /kaggle/working/train_items.pt y test_rows.json")


## 4. Script de entrenamiento (`train_ddp.py`)
Es el script oficial (RLCD con reglas de puntuación propias + entropía cruzada, en las dos GPU),
con cuatro cambios: `max_len` 2048, micro-batch 4 × acumulación 8, sobremuestreo de la clase
minoritaria después de apartar la porción de calibración, y el nombre del modelo `laya-codiesp`.

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 2048
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]

    # CodiEsp: oversample the minority answer of every question (e.g. noul 'true').
    # Done after the calibration hold-out, so no copy of a calibration item is trained on.
    # Deterministic (no randomness), so every rank builds exactly the same list.
    TARGET_RATIO = 0.35   # minority answers repeated up to ~35 % of the majority
    MAX_OVERSAMPLE = 5    # never more than 5 copies of the same item
    by_q = {}
    for it in train_items:
        by_q.setdefault(it["qid"], {}).setdefault(it["label"], []).append(it)
    extra = []
    for qid, groups in by_q.items():
        biggest = max(len(v) for v in groups.values())
        for lab, lst in groups.items():
            factor = min(MAX_OVERSAMPLE, max(1, round(biggest * TARGET_RATIO / len(lst))))
            if factor > 1:
                extra.extend(lst * (factor - 1))
                if rank == 0:
                    print(f"  oversample {qid} label {lab}: {len(lst)} x{factor}")
    train_items = train_items + extra
    my_items = train_items[rank::world_size]
    
    EPOCHS = 4
    MICRO_BATCH = 4      # 4 sequences per forward pass per GPU (2048 tokens on a T4)
    GRAD_ACCUM = 8       # Effective batch across 2 GPUs = 64 sequences (4 * 2 * 8)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-codiesp-v2"
        cfg["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Entrenar en las dos T4
Tras cada época se guarda `checkpoint_latest`, así que si Kaggle corta la sesión no se pierde todo.
El tiempo depende mucho de la longitud de los textos; vigila la salida: cada línea muestra época, paso y pérdida.

In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_codiesp_v2"
MODEL_DIR = model_dir

cmd = f"PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print("Ejecutando:", cmd)
!{cmd}


## 6. Evaluación en la partición `test` de CodiEsp (250 casos)
Mismas cifras que el informe de `TLayaEvaluator` en Delphi, más precisión y exhaustividad.
Para comparar: con el modelo base (troceado en Delphi) la exactitud fue diabetes 79,6 %,
hipertensión 59,2 %, cáncer 77,2 %, insuficiencia renal 51,2 % y tabaco 91,6 %.

In [ ]:
import json, time
import numpy as np
import pandas as pd
import laya

with open("/kaggle/working/test_rows.json", encoding="utf-8") as f:
    test_rows = json.load(f)

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

def predicted_label(qdef, ans):
    if qdef["type"] == "choice":
        return ans["choice"]
    if qdef["type"] == "noul":
        return "true" if ans["noul"] >= 0.5 else "false"
    probs = [ans["probabilities"].get(str(i), 0.0) for i in range(len(qdef.get("criteria", [])))]
    return str(int(np.argmax(probs)))

records = []
t0 = time.time()
for row in test_rows:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    res = agent_ft.predict(state, questions)["answers"]
    for qid, qdef in questions.items():
        ans = res[qid]
        pred = predicted_label(qdef, ans)
        prob = ans["noul"] if qdef["type"] == "noul" else ans["probabilities"].get(pred, 0.0)
        records.append({"id": row["id"], "question": qid, "gold": str(gold[qid]["label"]),
                        "pred": pred, "prob": prob})
print(f"Evaluados {len(test_rows)} casos en {time.time() - t0:.1f} s")

df = pd.DataFrame(records)
summary = []
for qid, g in df.groupby("question", sort=False):
    acc = (g.gold == g.pred).mean()
    majority = g.gold.value_counts(normalize=True).iloc[0]
    row = {"pregunta": qid, "casos": len(g), "exactitud": round(acc, 3),
           "siempre la mayoritaria": round(majority, 3)}
    positive = "true" if set(g.gold) <= {"true", "false"} else None
    if positive:
        tp = ((g.gold == "true") & (g.pred == "true")).sum()
        fp = ((g.gold == "false") & (g.pred == "true")).sum()
        fn = ((g.gold == "true") & (g.pred == "false")).sum()
        row["positivos"] = int((g.gold == "true").sum())
        row["encontrados (recall)"] = f"{tp}/{tp + fn}"
        row["aciertos al decir sí (precision)"] = f"{tp}/{tp + fp}"
    summary.append(row)
report = pd.DataFrame(summary)
print(report.to_markdown(index=False))

# --- calibración: ¿los errores llegan con menos seguridad que los aciertos? ---
df["confianza"] = np.where(df.pred == "true", df.prob,
                  np.where(df.pred == "false", 1 - df.prob, df.prob))
df["acierto"] = df.gold == df.pred
calib = []
for qid, g in df.groupby("question", sort=False):
    calib.append({"pregunta": qid,
                  "confianza media aciertos": round(g[g.acierto].confianza.mean(), 3),
                  "confianza media errores": round(g[~g.acierto].confianza.mean(), 3)
                      if (~g.acierto).any() else "-",
                  "casos con confianza < 0.8": int((g.confianza < 0.8).sum()),
                  "errores con confianza < 0.8": int(((g.confianza < 0.8) & ~g.acierto).sum())})
print("\nCalibración (la primera vuelta dio ~0.97 en aciertos y en errores):")
print(pd.DataFrame(calib).to_markdown(index=False))

print("\nTabaco (real -> predicha):")
t = df[df.question == "tabaco"]
print(pd.crosstab(t.gold, t.pred).to_markdown())

df.to_csv("/kaggle/working/codiesp_test_predictions.csv", index=False)
report.to_csv("/kaggle/working/codiesp_test_report.csv", index=False)


## 7. Empaquetar el modelo para descargarlo
Borra el punto de control intermedio (el modelo final está fuera de él), genera
`laya_codiesp_v2_model.zip` y muestra un enlace de descarga directa.
Si hiciste *Save & Run All (Commit)*, descárgalo desde la pestaña *Output* de la versión.

In [ ]:
import shutil, os
from IPython.display import FileLink

shutil.rmtree(os.path.join(OUTPUT_DIR, "checkpoint_latest"), ignore_errors=True)
zip_path = shutil.make_archive("/kaggle/working/laya_codiesp_v2_model", "zip", OUTPUT_DIR)
print(f"Modelo empaquetado: {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
print("Contenido:", os.listdir(OUTPUT_DIR))
os.chdir("/kaggle/working")
FileLink("laya_codiesp_v2_model.zip")
